In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")
# warnings.filterwarnings("W036")
import logging
logger = logging.getLogger("spacy")
logger.setLevel(logging.ERROR)
import pandas as pd

# ukb_target_pipe.py:70
# ukb_target_pipe.py:70
from Link_semmed_cuis import * #link_kg_concepts #*
from ukb_target_pipe  import *# make_target_df,model_features#*
from search_pubmed import run_search_pubmed #*

In [ ]:
## Cholelithiasis = gallstones
# TARGET_CODES_LIST = ("K80","K81","K82") #("K80","K56.3") # doesn't include other obstructionso r gallbladder diseases
# TARGET_CODES_LIST =("C15") # Oesophagus cancer. C15.9 = unspecifiedNote - may be in cause of death, not historical diagnoses.. - 1275 . Compare to: "Development and validation of a novel risk prediction algorithm to estimate 10-year risk of oesophageal cancer in primary care: prospective cohort study and evaluation of performance against two other risk prediction models"
# TARGET_CODES_LIST = ("M10","M1A") # Gout
# TARGET_CODES_LIST = ("K90") # Celiac/Coeliac disease
# TARGET_CODES_LIST = ("G35") # MS - multiple sclerosis.
## M40-M54 : Dorsopathy = diseases of the spine and paravertebral tissues (inc back pain!).
### https://www.aapc.com/codes/icd-10-codes/M51   # ,"M54.5" = lower back pain
# TARGET_CODES_LIST = ("M51.3","M51.2","M51.0","M51.1","M51.3","M47") #intervertebral disc degeneration(Lumbar is M51.36). M51.2 = disc displacement. M47 = Spondylosis
## heart - lots of subcats (chronic, angina, nonischemic (ischemia = blood flow), MI, IHF....; Myocardial infarction (MI) = "heart attack,"
## lacks cases of death due to that...
TARGET_CODES_LIST = ("I21.9") ## Heart attack
## Central retinal artery occlusion (Central retinal artery occlusion: H34.1) is a type of stroke that must be treated immediately. 
# TARGET_CODES_LIST = ("H34")#("H34.1") ## Retinal Vein Occlusion . (Central retinal artery occlusion: H34.1 = rare). Others - H34


In [ ]:
# # # Example configuration dictionary - gallstones
# config = {
#     # "TARGET_CODES_LIST" :("K80", "K81", "K82")  ## Cholelithiasis = gallstones 
#     "TARGET_CODES_LIST" :("H34") #  ## Retinal Vein Occlusion 
#     ,"do_IPW": True,
#     "do_boruta_fs" : False,
#     "K_IPW_RATIO":9,
#     'targets': [
#         "Cholelithiasis",
#         "Gallstone",
#         "Gallbladder disease",
#         "cholecystitis",
#         "Cholangitis"
#         ],
#     "FEATURES_REPORT_PATH":"gallstone_ipw_broad_feature_report.csv",
#     'QUERY_CANDIDATES_FILE': 'candidate_novel_cuis_chol.csv',
#     'DO_MINI_COMBINED_PATH_FILT': False
#     ,'SAVE_OUTPUTS': False,
#     "CANDIDATE_NOVEL_CUIS_FILEPATH": "candidate_novel_cuis_chol.csv", # output path for features/util
#     # "TARGET_NAME" :"GALLSTONES, Cholelithiasis",
#     # "additional_target_cui_terms_list" : ["C0008325", "C0008311"],
    
#     'OUTPUT_RES_PREFIX': 'gallstone_',
#     # "DIAG_TIDY_TABLE_PATH" : "../../df_ukbb_aux_tidy.parquet", #"../../ukbb-hack/df_diag_tidy.parquet",
#     }

# Fast_Run = False

In [ ]:
%%time
df = make_target_df(TARGET_CODES_LIST=TARGET_CODES_LIST,save_targ_df_only=True)
print(df.shape)

In [ ]:
print(df["y"].agg(["sum","mean","count"]).round(3))

In [ ]:
# df = df.iloc[:,0:8] # keep only some cols, add in rest as lookups? (faster upload) 

In [ ]:
df.select_dtypes("datetime").columns
## for sb parsing
for c in df.select_dtypes("datetime").columns:
    df[c] = df[c].dt.date
# df.select_dtypes("datetime")

In [ ]:
df

In [ ]:
df.columns[df.columns.str.contains("eight")]

In [ ]:
%%time
df = ipw_downsampling(df,K_IPW_RATIO=33,ipw_propensity_cols_list = [#'age',
                                                                    "YOB",
                                                                    'Sex',
                                                                    # 'age_X_sex',
                                                                    # 'Body mass index (BMI)(participant - p21001_i0)',
                                                                    # "Weight(participant - p21002_i0)",
                                                                'Weight (p21002)',
                                                                    # '(BMI) Body mass index (p21001)', # measured bmi +
                                                                    # '(BMI) Body mass index (p23104)'
                                                                    ])

df.shape

In [ ]:
# df.rename(columns={"y":config["TARGET_NAME"].replace(","," or")},inplace=True)

In [ ]:
# df.to_parquet("gallstone_ipw.parquet",index=False,)
# df.head(99).to_parquet("sample.parquet",index=False,)

In [ ]:
df.to_parquet(f'{"".join(TARGET_CODES_LIST)}_ipw.parquet',index=False,)